# Saddle Point Navigation with Newton MethodComparing four rotation control strategies for a 4-robot square formation.

In [ ]:
import numpy as npfrom itertools import combinationsclass SaddlePointNavigator:    def __init__(self, robot_distance=0.3, step_size=0.01, n_iterations=100,                  rotation_gain=0.0, rotation_mode='none'):        self.robot_distance = robot_distance        self.step_size = step_size        self.n_iterations = n_iterations        self.rotation_gain = rotation_gain        self.rotation_mode = rotation_mode        self.use_rotation = rotation_mode != 'none' and rotation_gain > 0        self.prev_newton_angle = None  # for quad_unwrap mode        self.history = {            'centroids': [], 'gradients': [], 'hessians': [],            'rotations': [], 'rotation_errors': [], 'newton_angles': [],        }            def scalar_field(self, x, y):        g1 = -((x + 2)**2 + y**2) / 2        g2 = -((x - 2)**2 + y**2) / 2        mx = np.maximum(g1, g2)        return mx + np.log(np.exp(g1 - mx) + np.exp(g2 - mx))        def initialize_robots(self, centroid, rotation_angle=None):        if rotation_angle is None:            rotation_angle = np.random.uniform(0, 2*np.pi)        angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2]) + rotation_angle        robots = np.array([[centroid[0] + self.robot_distance * np.cos(a),                            centroid[1] + self.robot_distance * np.sin(a)] for a in angles])        return robots, rotation_angle        def estimate_gradient_from_plane(self, points_3d):        A = np.column_stack([points_3d[:, 0], points_3d[:, 1], np.ones(3)])        coeffs = np.linalg.lstsq(A, points_3d[:, 2], rcond=None)[0]        return coeffs[0], coeffs[1]        def estimate_hessian(self, centroids, gradients):        A = np.column_stack([centroids[:, 0], centroids[:, 1], np.ones(4)])        try:            cx = np.linalg.lstsq(A, gradients[:, 0], rcond=None)[0]        except:            cx = [0, 0, 0]        try:            cy = np.linalg.lstsq(A, gradients[:, 1], rcond=None)[0]        except:            cy = [0, 0, 0]        return np.array([[cx[0], cx[1]], [cy[0], cy[1]]])        def compute_newton_step(self, gradient, hessian):        try:            det = np.linalg.det(hessian)            if abs(det) > 1e-10:                step = -np.linalg.solve(hessian, gradient)            else:                step = -np.linalg.pinv(hessian) @ gradient        except:            step = -gradient / (np.linalg.norm(gradient) + 1e-10)        if np.linalg.norm(step) > 1.0:            step = step / np.linalg.norm(step)        return step        def compute_rotation_error(self, hessian, current_rotation, gradient):        step = self.compute_newton_step(gradient, hessian)        if np.linalg.norm(step) > 1e-10:            newton_angle = np.arctan2(step[1], step[0])        else:            eigenvalues, eigenvectors = np.linalg.eigh(hessian)            principal_direction = eigenvectors[:, np.argmax(np.abs(eigenvalues))]            newton_angle = np.arctan2(principal_direction[1], principal_direction[0])                # Unwrap newton angle for quad_unwrap mode (one step of memory)        if self.rotation_mode == 'quad_unwrap' and self.prev_newton_angle is not None:            delta = newton_angle - self.prev_newton_angle            delta = np.arctan2(np.sin(delta), np.cos(delta))            newton_angle = self.prev_newton_angle + delta        if self.rotation_mode == 'quad_unwrap':            self.prev_newton_angle = newton_angle                raw_error = newton_angle - current_rotation                if self.rotation_mode == 'single':            error = np.arctan2(np.sin(raw_error), np.cos(raw_error))        elif self.rotation_mode in ('quad', 'quad_unwrap'):            error = (raw_error + np.pi/4) % (np.pi/2) - np.pi/4        else:            error = 0.0        return error        def navigate(self, start_centroid, initial_rotation=None):        centroid = np.array(start_centroid, dtype=float)        if initial_rotation is None:            current_rotation = np.random.uniform(0, 2*np.pi)        else:            current_rotation = initial_rotation                self.initial_centroid = centroid.copy()        self.initial_rotation = current_rotation                for iteration in range(self.n_iterations):            robots, _ = self.initialize_robots(centroid, current_rotation)            z_values = np.array([self.scalar_field(r[0], r[1]) for r in robots])            points_3d = np.column_stack([robots, z_values])                        gradients = []            centroids_list = []            for combo in combinations(range(4), 3):                grad = self.estimate_gradient_from_plane(points_3d[list(combo)])                gradients.append(grad)                centroids_list.append(np.mean(robots[list(combo)], axis=0))                        gradients = np.array(gradients)            centroids_arr = np.array(centroids_list)            hessian = self.estimate_hessian(centroids_arr, gradients)            avg_gradient = np.mean(gradients, axis=0)                        self.history['centroids'].append(centroid.copy())            self.history['gradients'].append(avg_gradient.copy())            self.history['hessians'].append(hessian.copy())            self.history['rotations'].append(current_rotation)                        direction = self.compute_newton_step(avg_gradient, hessian)            newton_angle = np.arctan2(direction[1], direction[0])            self.history['newton_angles'].append(newton_angle)                        if self.use_rotation:                rotation_error = self.compute_rotation_error(hessian, current_rotation, avg_gradient)                self.history['rotation_errors'].append(rotation_error)                current_rotation += self.rotation_gain * rotation_error                current_rotation = current_rotation % (2 * np.pi)                        centroid = centroid + self.step_size * direction                self.final_centroid = centroid        self.final_rotation = current_rotation        return centroiddef print_walkthrough(nav, n_steps=200):    kr = nav.rotation_gain    has_rot = nav.use_rotation    n = min(n_steps, len(nav.history['newton_angles']))        print(f'WALKTHROUGH: mode={nav.rotation_mode}, kr={kr} (first {n} steps)')    print('='*110)        if has_rot:        print(f"{'iter':>4}  {'x':>8}  {'y':>8}  {'newton_dir':>10}  {'d(newton)':>10}  {'error':>10}  {'update':>10}  {'theta':>10}")    else:        print(f"{'iter':>4}  {'x':>8}  {'y':>8}  {'newton_dir':>10}  {'d(newton)':>10}  {'theta':>10}")    print('-'*110)        for i in range(n):        cx, cy = nav.history['centroids'][i]        ndir = nav.history['newton_angles'][i]        theta = nav.history['rotations'][i]                if i > 0:            delta = ndir - nav.history['newton_angles'][i-1]            delta = np.arctan2(np.sin(delta), np.cos(delta))            delta_str = f'{delta:>+10.4f}'        else:            delta_str = f'{"---":>10}'                if has_rot:            err = nav.history['rotation_errors'][i]            upd = kr * err            print(f'{i:>4}  {cx:>+8.4f}  {cy:>+8.4f}  {ndir:>+10.4f}  {delta_str}  {err:>+10.4f}  {upd:>+10.4f}  {theta:>10.4f}')        else:            print(f'{i:>4}  {cx:>+8.4f}  {cy:>+8.4f}  {ndir:>+10.4f}  {delta_str}  {theta:>10.4f}')        print('-'*110)    fc = nav.final_centroid    gm = np.linalg.norm(nav.history['gradients'][-1])    print(f'Final: [{fc[0]:.4f}, {fc[1]:.4f}], dist={np.linalg.norm(fc):.6f}, |grad|={gm:.6f}')# ============================================================# Run all 3 versions# ============================================================print('\n' + '#'*60)print('# EXAMPLE 1: NO ROTATION (BASELINE)')print('#'*60 + '\n')np.random.seed(42)nav_none = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.0, rotation_mode='none')nav_none.navigate([1.0, -2.5])print_walkthrough(nav_none, 200)print('\n\n' + '#'*60)print('# EXAMPLE 2: SINGLE-ANGLE TRACKING (pi wrap)')print('#'*60 + '\n')np.random.seed(42)nav_single = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.1, rotation_mode='single')nav_single.navigate([1.0, -2.5])print_walkthrough(nav_single, 200)print('\n\n' + '#'*60)print('# EXAMPLE 3: QUAD SYMMETRY (pi/2 wrap)')print('#'*60 + '\n')np.random.seed(42)nav_quad = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.1, rotation_mode='quad')nav_quad.navigate([1.0, -2.5])print_walkthrough(nav_quad, 200)print('\n\n' + '#'*60)print('# EXAMPLE 4: QUAD + UNWRAP (pi/2 wrap, one step memory)')print('#'*60 + '\n')np.random.seed(42)nav_quad_uw = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.1, rotation_mode='quad_unwrap')nav_quad_uw.navigate([1.0, -2.5])print_walkthrough(nav_quad_uw, 200)# Summary comparisonprint('\n\n' + '#'*60)print('# SUMMARY COMPARISON')print('#'*60)for label, nav in [('No rotation', nav_none), ('Single (pi)', nav_single), ('Quad (pi/2)', nav_quad), ('Quad+unwrap', nav_quad_uw)]:    fc = nav.final_centroid    gm = np.linalg.norm(nav.history['gradients'][-1])    print(f'{label:>15}: final=[{fc[0]:>+8.4f}, {fc[1]:>+8.4f}]  dist={np.linalg.norm(fc):.6f}  |grad|={gm:.6f}')

In [ ]:
%matplotlib inline

## Example 1: No Rotation (Baseline)

In [ ]:
np.random.seed(42)nav_none = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.0, rotation_mode='none')nav_none.navigate([1.0, -2.5])nav_none.visualize('Example 1: No Rotation')print()print_walkthrough(nav_none, 200)

## Example 2: Single-Angle Tracking (pi wrap)

In [ ]:
np.random.seed(42)nav_single = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.1, rotation_mode='single')nav_single.navigate([1.0, -2.5])nav_single.visualize('Example 2: Single-Angle (pi wrap)')print()print_walkthrough(nav_single, 200)

## Example 3: Quad Symmetry (pi/2 wrap)

In [ ]:
np.random.seed(42)nav_quad = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.1, rotation_mode='quad')nav_quad.navigate([1.0, -2.5])nav_quad.visualize('Example 3: Quad Symmetry (pi/2 wrap)')print()print_walkthrough(nav_quad, 200)

## Example 4: Quad + Unwrap (pi/2 wrap, one step memory)

In [ ]:
np.random.seed(42)nav_quad_uw = SaddlePointNavigator(    robot_distance=0.25, step_size=0.02, n_iterations=1000,    rotation_gain=0.1, rotation_mode='quad_unwrap')nav_quad_uw.navigate([1.0, -2.5])nav_quad_uw.visualize('Example 4: Quad + Unwrap')print()print_walkthrough(nav_quad_uw, 200)

## Summary| Mode | Final Distance | Final |grad| | Outcome ||------|---------------|--------------|---------|| No rotation | 9.81 | 8.31 | Diverges || **Single (pi wrap)** | **0.20** | **0.20** | **Converges** || Quad (pi/2 wrap) | 0.81 | 0.81 | Limit cycle || Quad + unwrap | 0.82 | 0.82 | Limit cycle |### Key FindingThe single-angle (pi wrap) approach is the only mode that converges. The quad modes fail not because of Markov vs non-Markov, but because the pi/4 fundamental domain is too narrow: the estimation-rotation coupling produces errors near the boundary, and any perturbation flips the sign. The pi wrap has a boundary at pi which is never approached during normal navigation, so the controller tracks smoothly.### Rotation Modes**No rotation**: Fixed formation angle. Diverges because the fixed sample geometry gives degrading Hessian estimates as the field geometry rotates relative to the formation.**Single-angle (pi wrap)**: Track one robot arm toward the Newton direction. Error wrapped to [-pi, pi] via atan2. The wrap boundary is far from typical errors. Converges.**Quad symmetry (pi/2 wrap)**: Exploit 4-fold square symmetry. Error wrapped to [-pi/4, pi/4]. The tight domain means estimation-rotation coupling pushes errors to the boundary, causing sign-flip chatter. Limit cycle.**Quad + unwrap**: Unwrap the Newton angle before pi/2 mod. The mod operation nullifies the continuity benefit -- the narrow domain is the root cause, not the arctan2 discontinuity. Limit cycle.